In [1]:
import pandas as pd

In [2]:
#Using the segment as an additional predictor in an individual-level PD model.

numeric_features = [
    "Age",
    "Cibil Score",
    "Land acres",
    "NFCF",
    "Loan amount requested",
    "Existing Customer",
    "LTV"
]

categorical_features = [
    "Segment"
]


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_norm = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_norm = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder( handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_norm, numeric_features),
    ("categorical", categorical_norm, categorical_features)
])

In [4]:
#Splitting Data Training and Testing set
from sklearn.model_selection import train_test_split

model_df=pd.read_csv(
    "../data/processed/processed_data.csv"
)

X = model_df[numeric_features + categorical_features]
y = model_df["Default"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 101675
Testing records: 11298


In [5]:
#Building individual PD model
from sklearn.linear_model import LogisticRegression

pd_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

pd_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Age','Cibil Score','Land acres',...,'Existing Customer','LTV','Segment']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthr

In [6]:
pd_predictions = pd_model.predict_proba(X_test)[:, 1]

In [7]:
import joblib

joblib.dump(
    pd_model,
    "../src/models/underwriting_model.pkl"
)

['../src/models/underwriting_model.pkl']

In [8]:
pd_results = X_test.copy()


pd_results["Actual_Default"] = y_test.values

pd_results["Predicted_PD"] = pd_predictions

pd_results["Predicted_PD_%"] = ( pd_results["Predicted_PD"] * 100)

display(pd_results.head(5))

,Age,Cibil Score,Land acres,NFCF,Loan amount requested,Existing Customer,LTV,Segment,Actual_Default,Predicted_PD,Predicted_PD_%
36460,59,726,2,411922,760878,0,0.990021,4,0,0.007967,0.796748
34151,39,656,2,306231,860727,1,0.682788,2,0,0.035008,3.500802
91714,28,645,1,461574,451345,0,0.306922,1,0,0.037356,3.735601
56089,55,654,2,574087,869266,1,0.982445,2,0,0.004427,0.442662
98624,32,639,1,304676,500000,0,0.536492,1,0,0.047537,4.753665


In [9]:
def assign_risk_grade(pd):
    if pd < 0.05:
        return "A - Very Low Risk"
    elif pd < 0.10:
        return "B - Low Risk"
    elif pd < 0.20:
        return "C - Moderate Risk"
    elif pd < 0.30:
        return "D - High Risk"
    else:
        return "E - Very High Risk"


pd_results["Risk_Grade"] = (
    pd_results["Predicted_PD"]
    .apply(assign_risk_grade)
)

display(pd_results.head(5))

,Age,Cibil Score,Land acres,NFCF,Loan amount requested,Existing Customer,LTV,Segment,Actual_Default,Predicted_PD,Predicted_PD_%,Risk_Grade
36460,59,726,2,411922,760878,0,0.990021,4,0,0.007967,0.796748,A - Very Low Risk
34151,39,656,2,306231,860727,1,0.682788,2,0,0.035008,3.500802,A - Very Low Risk
91714,28,645,1,461574,451345,0,0.306922,1,0,0.037356,3.735601,A - Very Low Risk
56089,55,654,2,574087,869266,1,0.982445,2,0,0.004427,0.442662,A - Very Low Risk
98624,32,639,1,304676,500000,0,0.536492,1,0,0.047537,4.753665,A - Very Low Risk


In [10]:
#Model Evaluation_ROC-AUC
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(
    y_test,
    pd_predictions
)

print("ROC-AUC:", round(auc, 2))

ROC-AUC: 0.89


In [11]:
#Model Evaluation_Confusion matrix
from sklearn.metrics import confusion_matrix

threshold = 0.40

predicted_default = (
    pd_predictions >= threshold
).astype(int)

cm = confusion_matrix(
    y_test,
    predicted_default
)

print(cm)

[[10869    39]
 [  358    32]]


In [12]:


#Model Evaluation_Classification report

from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        predicted_default
    )
)

              precision    recall  f1-score   support

           0       0.97      1.00      0.98     10908
           1       0.45      0.08      0.14       390

    accuracy                           0.96     11298
   macro avg       0.71      0.54      0.56     11298
weighted avg       0.95      0.96      0.95     11298



In [13]:
#Default precision of only 7%.
#Conducting threshold analysis

threshold_results = []

thresholds = [
    0.05, 0.10, 0.15, 0.20,
    0.25, 0.30, 0.35, 0.40,
    0.45, 0.50, 0.60, 0.70
]

for threshold in thresholds:

    predictions = (
        pd_predictions >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix( y_test,predictions).ravel()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    f1 = (  2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0 )

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "False_Positive_Rate": fpr,
        "F1": f1,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

threshold_df = pd.DataFrame(threshold_results)

for col in [
    "Precision",
    "Recall",
    "Specificity",
    "False_Positive_Rate",
    "F1"
]:
    threshold_df[col] = (threshold_df[col] * 100).round(2)

display(threshold_df)


,Threshold,Precision,Recall,Specificity,False_Positive_Rate,F1,TP,FP,FN,TN
0,0.05,14.00,75.90,83.32,16.68,23.63,296,1819,94,9089
1,0.10,21.20,57.18,92.40,7.60,30.93,223,829,167,10079
2,0.15,26.43,41.54,95.87,4.13,32.30,162,451,228,10457
3,0.20,30.54,28.97,97.64,2.36,29.74,113,257,277,10651
4,0.25,31.28,19.49,98.47,1.53,24.01,76,167,314,10741
5,0.30,33.94,14.36,99.00,1.00,20.18,56,109,334,10799
6,0.35,39.81,10.51,99.43,0.57,16.63,41,62,349,10846
7,0.40,45.07,8.21,99.64,0.36,13.88,32,39,358,10869
8,0.45,51.22,5.38,99.82,0.18,9.74,21,20,369,10888
9,0.50,52.94,2.31,99.93,0.07,4.42,9,8,381,10900


In [14]:
#Model Evaluation_Confusion matrix
from sklearn.metrics import confusion_matrix

threshold = 0.10

predicted_default = (
    pd_predictions >= threshold
).astype(int)



from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        predicted_default
    )
)

              precision    recall  f1-score   support

           0       0.98      0.92      0.95     10908
           1       0.21      0.57      0.31       390

    accuracy                           0.91     11298
   macro avg       0.60      0.75      0.63     11298
weighted avg       0.96      0.91      0.93     11298



In [15]:
#Model Evaluation_PD calibration

pd_results["PD_Band"] = pd.cut(
    pd_results["Predicted_PD"],
    bins=[0, 0.05, 0.10, 0.20, 0.30, 1],
    labels=[
        "0-5%",
        "5-10%",
        "10-20%",
        "20-30%",
        "30%+"
    ],
    include_lowest=True
)

calibration = (
    pd_results
    .groupby("PD_Band", observed=False)
    .agg(
        Customers=("Actual_Default", "count"),
        Actual_Defaults=("Actual_Default", "sum"),
        Actual_Default_Rate=("Actual_Default", "mean"),
        Average_Predicted_PD=("Predicted_PD", "mean")
    )
    .reset_index()
)

calibration["Actual_Default_Rate"] *= 100
calibration["Average_Predicted_PD"] *= 100

display(calibration)

,PD_Band,Customers,Actual_Defaults,Actual_Default_Rate,Average_Predicted_PD
0,0-5%,9183,94,1.023631,0.998797
1,5-10%,1063,73,6.867357,6.985118
2,10-20%,682,110,16.129032,13.931702
3,20-30%,205,57,27.804878,24.259290
4,30%+,165,56,33.939394,39.817083


In [16]:
#Model Evaluation
total_customers = len(y_test)
total_defaults = y_test.sum()

overall_default_rate = (total_defaults / total_customers) * 100
overall_predicted_pd = pd_predictions.mean() * 100

print(f"Total Customers: {total_customers:,}")
print(f"Total Defaults: {total_defaults:,}")
print(f"Overall Actual Default Rate: {overall_default_rate:.2f}%")
print(f"Overall Average Predicted PD: {overall_predicted_pd:.2f}%")

Total Customers: 11,298
Total Defaults: 390
Overall Actual Default Rate: 3.45%
Overall Average Predicted PD: 3.33%


In [17]:
def assign_rate(pd):
    if pd < 0.05:
        return 0.135
    elif pd < 0.10:
        return 0.140
    elif pd < 0.20:
        return 0.145
    elif pd < 0.30:
        return 0.15
    else:
        return 0.155


pd_results["rate"] = (
    pd_results["Predicted_PD"]
    .apply(assign_rate)
)

display(pd_results.head(5))

,Age,Cibil Score,Land acres,NFCF,Loan amount requested,Existing Customer,LTV,Segment,Actual_Default,Predicted_PD,Predicted_PD_%,Risk_Grade,PD_Band,rate
36460,59,726,2,411922,760878,0,0.990021,4,0,0.007967,0.796748,A - Very Low Risk,0-5%,0.135
34151,39,656,2,306231,860727,1,0.682788,2,0,0.035008,3.500802,A - Very Low Risk,0-5%,0.135
91714,28,645,1,461574,451345,0,0.306922,1,0,0.037356,3.735601,A - Very Low Risk,0-5%,0.135
56089,55,654,2,574087,869266,1,0.982445,2,0,0.004427,0.442662,A - Very Low Risk,0-5%,0.135
98624,32,639,1,304676,500000,0,0.536492,1,0,0.047537,4.753665,A - Very Low Risk,0-5%,0.135


In [19]:
kmeans = joblib.load("../src/models/kmeans_model.pkl")
scaler2 = joblib.load("../src/models/segmentation_scaler.pkl")

new_customer_seg = pd.DataFrame({
    "NFCF": [507600],
    "Land acres": [8],
    "Loan amount requested": [839966],
    "LTV": [0.60],
    "Existing Customer":[1]
})

#  Transform new customer using same scaler
new_customer_seg_scaled = scaler2.transform(new_customer_seg)

# Predict segment
new_segment = kmeans.predict(new_customer_seg_scaled)[0]

segment_names = {
    0: "Existing customer - Low Exposure",
    1: "New customer- Low Exposure",
    2: "Existing customer - High Exposure",
    3: "Strong Financial Profile",
    4: "New customer- High Exposure"
}

segment_name = segment_names[new_segment]

print("Customer Segment:", segment_name)

new_customer = pd.DataFrame({
    "Age": [32],
    "Cibil Score": [800],
    "NFCF": [507600],
    "Land acres": [8],
    "Loan amount requested": [500000],
    "Loan_tenure(years)": [4],
    "LTV": [0.60],
    "Existing Customer":[1],
    "Segment": new_segment
})

new_pd = pd_model.predict_proba(new_customer)[:, 1][0]

print("Predicted PD:", round(new_pd * 100, 2), "%")

risk_grade = assign_risk_grade(new_pd)

print("Risk Grade:", risk_grade)

rate = assign_rate(new_pd)

print("Rate:", rate)

AI = (new_customer["Loan amount requested"]*rate*(1 + rate)**new_customer["Loan_tenure(years)"]) / ((1 + rate)**new_customer["Loan_tenure(years)"]- 1)

NFCF_AI=((new_customer["NFCF"])/AI)

print("NFCF:AI:", round(NFCF_AI.iloc[0],2))

#Underwriting Decision
def underwriting_decision(new_pd,NFCF_AI):

    if new_pd < 0.05 and NFCF_AI >= 1.5 :
        return "APPROVE"

    elif new_pd < 0.30 and NFCF_AI >= 1.1:
        return "REFER TO CREDIT"

    else:
        return "REJECT"


decision = underwriting_decision(new_pd,NFCF_AI.iloc[0])

print("Underwriting Decision:", decision)

Customer Segment: Strong Financial Profile
Predicted PD: 0.0 %
Risk Grade: A - Very Low Risk
Rate: 0.135
NFCF:AI: 2.99
Underwriting Decision: APPROVE
